# 🍜 STEP 02 — 단일 에이전트: ChatCompletionAgent

**STEP 01**에서 만든 Plugin을 `ChatCompletionAgent`에 장착해
F&B 전문 에이전트 3종을 각각 단독으로 구현합니다.

---

## 이 노트북에서 배울 것

| 개념 | 설명 |
|------|------|
| `ChatCompletionAgent` | 역할·지시문·플러그인이 있는 전문 에이전트 |
| `agent.get_response()` | 단일 응답 요청 |
| `agent.invoke_stream()` | 스트리밍 응답 (실시간 출력) |
| `AgentThread` | 에이전트별 대화 상태 유지 |

---

## 이 단계에서 만들 에이전트
```
┌─────────────────────┐  ┌──────────────────────┐  ┌────────────────────────┐
│   LegalTaxAgent     │  │   LocationAgent      │  │   FinanceAgent         │
│ 법령·세무 안내       │  │ 상권 분석            │  │ 재무 리스크 분석       │
│ FnbLegalPlugin      │  │ FnbLocationPlugin    │  │ FnbFinancePlugin       │
└─────────────────────┘  └──────────────────────┘  └────────────────────────┘
```

In [1]:
import asyncio, os, json
from typing import Annotated
from dotenv import load_dotenv

from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureChatPromptExecutionSettings
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.functions import kernel_function, KernelArguments
from semantic_kernel.contents import ChatHistory

load_dotenv()
print("임포트 완료")

임포트 완료


---
## 1. 헬퍼: Kernel 팩토리 함수

> **왜 에이전트마다 별도 Kernel을 만드나?**
> 
> 각 에이전트에게 다른 플러그인 세트만 노출시키기 위해서입니다.
> LegalTaxAgent는 법령 플러그인만, LocationAgent는 지도 플러그인만 갖습니다.

In [2]:
def make_kernel() -> Kernel:
    """기본 Azure OpenAI 서비스가 등록된 빈 Kernel을 반환합니다."""
    kernel = Kernel()
    kernel.add_service(
        AzureChatCompletion(
            deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
            endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-03-01-preview"),
        )
    )
    return kernel

def make_auto_settings():
    s = AzureChatPromptExecutionSettings()
    s.function_choice_behavior = FunctionChoiceBehavior.NoneInvoke()
    return s

print("헬퍼 함수 정의 완료")

헬퍼 함수 정의 완료


---
## 2. 플러그인 정의 (STEP 01 복습 + FinancePlugin 추가)

In [3]:
# ─────────────────────────────────────────
# 법령·세무 플러그인
# ─────────────────────────────────────────
class FnbLegalPlugin:
    @kernel_function(
        description="F&B 업종별 영업 인허가 요건과 필요 서류를 반환합니다."
    )
    def get_license_requirements(
        self,
        business_type: Annotated[str, "업종 (예: 일반음식점, 카페)"],
        area_sqm: Annotated[float, "영업 면적 (제곱미터)"],
    ) -> str:
        docs = {
            "일반음식점": ["영업신고(보건소)", "위생교육(6h)", "소방완비증명(300㎡↑)"],
            "카페": ["영업신고(보건소)", "위생교육(3h)"],
        }
        items = docs.get(business_type, ["해당 정보 없음"])
        if business_type == "일반음식점" and area_sqm >= 300:
            items.append("소방완비증명서 필수")
        return "\n".join(items)

    @kernel_function(description="예상 연매출에 따른 간이/일반 과세유형을 안내합니다.")
    def get_tax_type_guide(
        self,
        annual_revenue_krw: Annotated[int, "예상 연 매출액 (원 단위)"],
    ) -> str:
        if annual_revenue_krw < 104_000_000:
            return "간이과세자 대상 (연 매출 1억 400만원 미만) — 부가세 부담 낮으나 세금계산서 발행 불가"
        return "일반과세자 대상 — 부가세 10%, 세금계산서 발행 가능"


# ─────────────────────────────────────────
# 상권 분석 플러그인
# ─────────────────────────────────────────
class FnbLocationPlugin:
    @kernel_function(
        description="위치와 업종에 따른 상권 분석(유동인구, 경쟁업체, 임대료)을 반환합니다."
    )
    def analyze_commercial_area(
        self,
        location: Annotated[str, "분석할 위치"],
        business_type: Annotated[str, "창업 업종"],
    ) -> str:
        # Mock — 실제 구현: 카카오 Local API 호출
        return json.dumps(
            {
                "location": location,
                "daily_floating_population": 45000,
                "competitor_count_500m": 12,
                "avg_monthly_rent_krw_per_sqm": 85000,
                "commercial_rating": "A",
            },
            ensure_ascii=False,
        )


# ─────────────────────────────────────────
# 재무 분석 플러그인 (신규)
# ─────────────────────────────────────────
class FnbFinancePlugin:
    @kernel_function(
        description="Calculates break-even point using initial investment, fixed costs, avg revenue per customer, and variable cost ratio."
    )
    def calculate_bep(
        self,
        initial_investment_krw: Annotated[
            int, "Total initial investment amount in KRW"
        ],
        monthly_fixed_cost_krw: Annotated[
            int, "Monthly fixed costs (rent, labor, etc.) in KRW"
        ],
        avg_revenue_per_customer_krw: Annotated[
            int, "Average revenue per customer in KRW"
        ],
        variable_cost_ratio: Annotated[
            float, "Variable cost ratio between 0 and 1 (e.g. 0.35 for 35%)"
        ],
    ) -> str:
        # 공헌이익률 = 1 - 변동비율
        contribution_margin_ratio = 1.0 - variable_cost_ratio
        # 월 BEP 매출 = 월 고정비 / 공헌이익률
        monthly_bep_revenue = monthly_fixed_cost_krw / contribution_margin_ratio
        # 월 BEP 고객 수 = 월 BEP 매출 / 객단가
        monthly_bep_customers = monthly_bep_revenue / avg_revenue_per_customer_krw
        # 투자 회수 기간 (개월)
        monthly_profit_at_bep_plus_20pct = (
            monthly_bep_revenue * 0.2 * contribution_margin_ratio
        )
        payback_months = (
            initial_investment_krw / monthly_profit_at_bep_plus_20pct
            if monthly_profit_at_bep_plus_20pct > 0
            else float("inf")
        )

        return (
            "monthly_bep_revenue: %s KRW, "
            "monthly_bep_customers: %d persons, "
            "daily_bep_customers: %d persons, "
            "payback_months: %.1f months"
            % (
                "{:,.0f}".format(monthly_bep_revenue),
                int(monthly_bep_customers),
                int(monthly_bep_customers / 30),
                payback_months,
            )
        )


print("플러그인 3종 정의 완료")

플러그인 3종 정의 완료


---
## 3. 전문 에이전트 3종 생성

> **`ChatCompletionAgent` 구조**:
> ```
> ChatCompletionAgent(
>     name         = 에이전트 이름 (GroupChat에서 식별자)
>     description  = 이 에이전트가 무엇을 하는지 (트리아지 에이전트가 라우팅 참고)
>     instructions = 시스템 프롬프트 (역할 + 행동 규칙)
>     kernel       = Plugin이 등록된 Kernel
>     arguments    = 실행 설정 (FunctionChoiceBehavior 등)
> )
> ```

In [4]:
# ── 에이전트 1: 법령·세무 전문 ──
legal_kernel = make_kernel()
legal_kernel.add_plugin(FnbLegalPlugin(), plugin_name="FnbLegal")

legal_tax_agent = ChatCompletionAgent(
    name="LegalTaxAgent",
    description="F&B 창업 관련 인허가 절차, 위생 교육, 영업신고, 과세유형 등 법령·세무 전문 에이전트",
    instructions=(
        "당신은 F&B 창업 법령·세무 전문가입니다.\n"
        "식품위생법, 상가임대차보호법, 근로기준법, 부가가치세법에 기반하여 정확한 정보를 제공합니다.\n"
        "모든 답변에는 관련 법조항(예: 식품위생법 제37조)을 명시하세요.\n"
        "불확실한 법률 해석은 '관할 기관에 확인 권장'으로 안내하세요."
    ),
    kernel=legal_kernel,
    arguments=KernelArguments(settings=make_auto_settings()),
)

# ── 에이전트 2: 상권 분석 전문 ──
location_kernel = make_kernel()
location_kernel.add_plugin(FnbLocationPlugin(), plugin_name="FnbLocation")

location_agent = ChatCompletionAgent(
    name="LocationAgent",
    description="유동인구, 경쟁업체 분포, 카카오 지도 기반 상권 분석 전문 에이전트",
    instructions=(
        "당신은 F&B 창업 상권 분석 전문가입니다.\n"
        "카카오 지도 API와 소상공인시장진흥공단 데이터를 기반으로 분석합니다.\n"
        "유동인구, 경쟁업체 수, 임대료 수준을 수치와 함께 제공하세요.\n"
        "분석 결과는 '리스크 / 기회 요소' 형태로 구조화하여 답하세요."
    ),
    kernel=location_kernel,
    arguments=KernelArguments(settings=make_auto_settings()),
)

# ── 에이전트 3: 재무 분석 전문 ──
finance_kernel = make_kernel()
# finance_kernel.add_plugin(FnbFinancePlugin(), plugin_name="FnbFinance")

finance_agent = ChatCompletionAgent(
    name="FinanceAgent",
    description="BEP, scenario analysis, risk simulation specialist",
    instructions=(
        "You are an F&B startup finance analysis expert. "
        "Provide BEP calculation, scenario-based revenue forecast, and risk assessment. "
        "Always include optimistic, base, and pessimistic scenarios. "
        "Respond in Korean, but use English internally for calculations."
    ),
    kernel=finance_kernel,
    # arguments=KernelArguments(settings=make_auto_settings()),
)

print("에이전트 3종 생성 완료!")
print("- LegalTaxAgent:", legal_tax_agent.name)
print("- LocationAgent:", location_agent.name)
print("- FinanceAgent: ", finance_agent.name)

에이전트 3종 생성 완료!
- LegalTaxAgent: LegalTaxAgent
- LocationAgent: LocationAgent
- FinanceAgent:  FinanceAgent


---
## 4. 단일 응답 테스트 — `get_response()`

> 한 번의 질문에 한 번의 응답을 받는 가장 단순한 패턴입니다.

In [5]:
# LegalTaxAgent 단독 테스트
response = await legal_tax_agent.get_response(
    messages="서울 강남구에서 30평짜리 일반음식점을 오픈하려 합니다. 필요한 인허가 절차를 알려주세요."
)
print("[LegalTaxAgent 응답]")
print(response.content)

[LegalTaxAgent 응답]
일반음식점을 오픈하기 위해서는 식품위생법에 따라 여러 가지 인허가 절차를 밟아야 합니다. 서울 강남구에서 30평(약 99.17제곱미터) 일반음식점을 오픈할 경우 필요한 인허가 요건과 필요 서류를 알려드리겠습니다. 잠시만 기다려 주세요.


In [6]:
# LocationAgent 단독 테스트
response2 = await location_agent.get_response(
    messages="서울 마포구 홍대입구역 근처에서 카페 창업을 고려 중입니다. 상권 분석을 해주세요."
)
print("[LocationAgent 응답]")
print(response2.content)

[LocationAgent 응답]
상권 분석을 위해 홍대입구역 근처의 카페 업종에 대한 유동인구, 경쟁업체 수, 임대료 수준 등을 조사하겠습니다. 잠시만 기다려 주세요.


In [7]:
# FinanceAgent 단독 테스트
response3 = await finance_agent.get_response(
    messages=(
        "카페 창업을 계획 중입니다. "
        "초기 투자비 8천만원, 월 고정비 350만원, 객단가 7000원, 재료비율 35%로 "
        "손익분기점을 계산해 주세요."
    )
)
print("[FinanceAgent 응답]")
print(response3.content)

[FinanceAgent 응답]
카페 창업 손익분기점(BEP)을 계산하겠습니다.

1. **BEP 계산**  
   - 초기 투자비: 80,000,000원
   - 월 고정비: 3,500,000원
   - 객단가: 7,000원
   - 재료비율: 35% (재료비로 2,450원)  

### BEP(판매량) = 고정비 / (판매가격 - 변동비)

고정비: 8,000,000원 (초기 투자 포함 계산 위해 한번에 계산)  
변동비: 2,450원  

```  
BEP 판매량 = 고정비 / (객단가 - 변동비)  
BEP 판매량 = 3,500,000 / (7,000 - 2,450)  
BEP 판매량 = 3,500,000 / 4,550 ≈ 769.23
```

회수 기간을 포함한 초기 투자비를 감안한 월 BEP 매출량은 약 770잔 이상입니다.

2. **시나리오 기반 수익 예측**  
   - **Optimistic 시나리오**  
     - 고객 수가 20% 증가하여 월 평균 924잔 판매 (770 * 1.2)
     - 월 매출: 924 * 7,000 = 6,468,000원  
     - 월 수익 = 6,468,000 - (924 * 2,450) - 3,500,000 = 2,246,200원

   - **Base 시나리오**  
     - Monthly sales: 770잔 유지  
     - 월 매출: 770 * 7,000 = 5,390,000원  
     - 월 수익 = 5,390,000 - (770 * 2,450) - 3,500,000 = 1,096,500원

   - **Pessimistic 시나리오**  
     - 고객 수가 20% 감소하여 월 평균 616잔 판매 (770 * 0.8)
     - 월 매출: 616 * 7,000 = 4,312,000원  
     - 월 수익 = 4,312,000 - (616 * 2,450) - 3,500,000 = -78,200원

3. **리스크 평가**  
- Optimistic 시나리오에서는 수익이 

---
## 5. 멀티턴 대화 — `AgentThread` 로 대화 상태 유지

> `AgentThread`는 에이전트와의 대화 세션입니다.
> 같은 thread를 계속 전달하면 이전 대화 내용을 기억합니다.
>
> STEP 01의 핸드오프 오케스트레이터에서 `thread_checkpoint_store`가 바로 이 역할!

In [8]:
# 완전히 새 셀 — 기존 코드 무관하게 테스트
import asyncio, os
from dotenv import load_dotenv
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion

load_dotenv()

kernel = Kernel()
kernel.add_service(AzureChatCompletion(
    deployment_name=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"),
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
))

agent = ChatCompletionAgent(
    name="TestAgent",
    instructions="You are a helpful assistant.",
    kernel=kernel,
)

# 영어로만 테스트
response = await agent.get_response(messages="Hello!")
print(response.content)

Hello! How can I assist you today?


In [9]:
from semantic_kernel.agents import ChatHistoryAgentThread

import logging
logging.basicConfig(level=logging.DEBUG)

async def multi_turn_demo():
    """FinanceAgent와 멀티턴 대화 — 이전 맥락을 기억합니다."""

    # thread = 이 에이전트와의 대화 세션
    thread = ChatHistoryAgentThread()

    questions = [
        "초기 투자비 8천만원, 월 고정비 350만원, 객단가 7000원, 재료비율 35%로 BEP 계산해줘.",
        "방금 계산한 BEP 기준으로, 하루에 몇 명이 와야 손익분기점을 넘길 수 있어?",  # 이전 대화 기억해야 답 가능
        "만약 객단가를 9000원으로 올리면 BEP가 어떻게 달라져?",  # 연속 맥락
    ]

    for i, q in enumerate(questions, 1):
        print("\n" + "="*60)
        print("[질문 %d] %s" % (i, q))
        print("-"*60)

        response = await finance_agent.get_response(
            messages=q,
            thread=thread,  # 같은 thread 전달 → 이전 대화 기억
        )
        # thread는 get_response 호출 후 자동 업데이트됨
        thread = response.thread
        print(response.content)

await multi_turn_demo()


# 파인튜닝한 모델에서 function calling이 안되는 문제발견
# 뭔가수정을해야할듯?

DEBUG:semantic_kernel.prompt_template.kernel_prompt_template:Rendering list of 1 blocks
DEBUG:semantic_kernel.prompt_template.kernel_prompt_template:Rendered prompt: You are an F&B startup finance analysis expert. Provide BEP calculation, scenario-based revenue forecast, and risk assessment. Always include optimistic, base, and pessimistic scenarios. Respond in Korean, but use English internally for calculations.
DEBUG:semantic_kernel.agents.chat_completion.chat_completion_agent:[ChatCompletionAgent] Invoking AzureChatCompletion.
DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'api-key': '<redacted>'}, 'files': None, 'idempotency_key': 'stainless-python-retry-c97a1ec3-5416-4063-8f3a-4f10d2aabf59', 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': 'You are an F&B startup finance analysis expert. Provide BEP calculation, scenario-based revenue forecast, and risk assessment. Always include optimistic, base, and 


[질문 1] 초기 투자비 8천만원, 월 고정비 350만원, 객단가 7000원, 재료비율 35%로 BEP 계산해줘.
------------------------------------------------------------


DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001CBC9115B40>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x000001CBC9153540> server_hostname='student02-11-1604-resource.cognitiveservices.azure.com' timeout=5.0
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001CBC9116560>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 400, b'Bad Request', [(b'Content-Length', b'620'), (b'Content-Type', b'application/json'), (b'apim-request-id', b'fd6a64a6-f9bc-4d7d-904f

ContentFilterAIException: <class 'semantic_kernel.connectors.ai.open_ai.services.azure_chat_completion.AzureChatCompletion'> service encountered a content error

---
## 6. 스트리밍 응답 — `invoke_stream()`

> 실제 서비스에서는 `invoke_stream()`으로 토큰을 실시간 출력합니다.
> FastAPI에서 `StreamingResponse`로 프론트에 전달하는 방식입니다.

In [10]:
async def streaming_demo():
    """LegalTaxAgent 스트리밍 응답 — 토큰이 생성되는 즉시 출력."""
    print("[LegalTaxAgent 스트리밍 응답]")
    print("-" * 50)

    async for chunk in legal_tax_agent.invoke_stream(
        messages="상가임대차보호법에서 임차인이 꼭 알아야 할 핵심 권리 3가지를 설명해줘."
    ):
        # chunk.content: 이번에 생성된 토큰 조각
        print(chunk.content, end="", flush=True)

    print()  # 줄바꿈

await streaming_demo()

DEBUG:semantic_kernel.prompt_template.kernel_prompt_template:Rendering list of 1 blocks
DEBUG:semantic_kernel.prompt_template.kernel_prompt_template:Rendered prompt: 당신은 F&B 창업 법령·세무 전문가입니다.
식품위생법, 상가임대차보호법, 근로기준법, 부가가치세법에 기반하여 정확한 정보를 제공합니다.
모든 답변에는 관련 법조항(예: 식품위생법 제37조)을 명시하세요.
불확실한 법률 해석은 '관할 기관에 확인 권장'으로 안내하세요.
DEBUG:semantic_kernel.agents.chat_completion.chat_completion_agent:[ChatCompletionAgent] Invoking AzureChatCompletion.
DEBUG:semantic_kernel.agents.chat_completion.chat_completion_agent:[ChatCompletionAgent] Invoked AzureChatCompletion with message count: 2.
DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'api-key': '<redacted>'}, 'files': None, 'idempotency_key': 'stainless-python-retry-b637b0f9-add5-4f48-94f8-0c98a996ff4f', 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': "당신은 F&B 창업 법령·세무 전문가입니다.\n식품위생법, 상가임대차보호법, 근로기준법, 부가가치세법에 기반하여 정확한 정보를 제공합니다.\n모든 답변에는 관련 법조항(예: 식품위생법 제37조)을 명시하세요.\n불확실한 법

[LegalTaxAgent 스트리밍 응답]
--------------------------------------------------


DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001CBC92734F0>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x000001CBC91534C0> server_hostname='student02-11-1604-resource.cognitiveservices.azure.com' timeout=5.0
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.anyio.AnyIOStream object at 0x000001CBC92BEB30>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Transfer-Encoding', b'chunked'), (b'Content-Type', b'text/event-stream; charset=utf-8'), (b'apim-request-id', b'c668ebbb-

상가임대차보호법은 상가 임차인의 권리를 보호하기 위해 마련된 법률로, 임차인이 꼭 알아야 할 핵심 권리 3가지는 다음과 같습니다.

1. **계약 갱신요구권 (제10조)**: 임차인은 계약 기간이 만료되기 전 갱신을 요구할 수 있으며, 임대인은 특별한 사유가 없는 한 이를 거부할 수 없습니다. 이 권리는 임차인이 영업을 계속할 수 있도록 보장하기 위한 것으로, 임대차 계약을 처음 체결한 날로부터 총 10년을 초과할 수 없습니다.

2. **임대차보호 범위 (제2조 및 제3조)**: 상가임대차보호법의 적용을 받기 위해서는 보증금이 일정한 기준 금액 이하이어야 합니다. 임차인의 계약 조건이 법의 적용 범위에 해당하는 경우 보증금 보호, 계약 갱신 요구 등 다양한 법적 보호를 받습니다.

3. **차임증감청구권 (제11조)**: 경제 상황의 변동, 공가 상승 등으로 인해 차임이나 보증금의 적정성이 변한 경우 임차인은 법원의 판단을 받아 이를 조정할 수 있습니다. 즉, 임대료가 과도하게 인상되거나 감소해야 하는 상황에서는 법적 절차를 통해 적정한 수준으로 조정할 수 있는 권리를 가집니다.

이러한 권리는 임차인의 사업 안정성과 권리를 보호하기 위해 중요한 역할을 하며, 법적 분쟁 시 상가임대차보호법을 근거로 자신을 보호할 수 있는 수단이 됩니다. 각 상황에 따라 적용이 다를 수 있으므로 구체적인 사안에 대해서는 관할 기관에 확인하는 것이 좋습니다.

DEBUG:httpcore.http11:response_closed.started
DEBUG:httpcore.http11:response_closed.complete


DEBUG:httpcore.http11:receive_response_body.failed exception=GeneratorExit()


---
## 7. ✅ STEP 02 정리

| 배운 것 | 코드 패턴 | 다음 단계 연결 |
|---------|-----------|---------------|
| 전문 에이전트 생성 | `ChatCompletionAgent(name, instructions, kernel)` | STEP 03 GroupChat 참여자 |
| 단일 응답 | `agent.get_response(messages)` | 기본 호출 패턴 |
| 멀티턴 대화 | `thread = ChatHistoryAgentThread()` | 세션 관리의 기초 |
| 스트리밍 | `async for chunk in agent.invoke_stream()` | FastAPI StreamingResponse |

---

### 다음 STEP 미리보기
```
STEP 03: AgentGroupChat (멀티 에이전트)
  - 3개 에이전트를 하나의 그룹으로 묶기
  - KernelFunctionSelectionStrategy: 어떤 에이전트가 다음에 응답할지 결정
  - KernelFunctionTerminationStrategy: 언제 대화를 끝낼지 결정
  - 트리아지 에이전트(오케스트레이터) 패턴
```